# Model Training

This notebook demonstrates the model training process with MLflow tracking.

In [ ]:
import sys
sys.path.append('..')

import torch
import mlflow
import mlflow.pytorch
from src.models.architecture import get_model
from src.data.data_loader import get_data_loaders

## Setup MLflow

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("cats_vs_dogs_classification")
print("MLflow experiment set up successfully")

## Load Data

In [ ]:
# Get data loaders
train_loader, val_loader, test_loader = get_data_loaders()

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Initialize Model

In [ ]:
# Get model
model = get_model("cnn")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Print model info
info = model.get_model_info()
print(f"Model: {info['model_name']}")
print(f"Total parameters: {info['total_parameters']:,}")

## Training Loop

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 5  # Reduced for notebook demonstration

# Start MLflow run
with mlflow.start_run():
    mlflow.log_params({
        "learning_rate": 0.001,
        "epochs": epochs,
        "optimizer": "adam"
    })
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_accuracy = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss/len(train_loader):.4f}, Accuracy: {train_accuracy:.2f}%")
        
        # Log metrics
        mlflow.log_metrics({
            "train_loss": train_loss/len(train_loader),
            "train_accuracy": train_accuracy
        }, step=epoch)
    
    # Log model
    mlflow.pytorch.log_model(model, "model")

print("Training completed!")